# AI-Powered Trading Assistant
## ITAI 2376 -- Final Notebook

**Student:** Javon Darby  
**Institution:** Houston City College  
**Course:** ITAI 2376 -- Artificial Intelligence Capstone

This notebook implements an AI-powered trading assistant that combines a deep
learning sentiment classifier (FinBERT) with a large language model reasoning
agent (via OpenRouter) to analyze stocks and provide structured investment
guidance for beginner investors.

## System Architecture

The system consists of three cooperating components:

```
+-----------------+       +------------------+       +------------------+
|   User Query    | ----> |  LangChain Agent | ----> |  LLM Brain       |
|                 |       |  (Orchestrator)  |       |  (OpenRouter)    |
+-----------------+       +--------+---------+       +--------+---------+
                                   |                          |
                          +--------v---------+       +--------v---------+
                          | Stock Price Tool  |       | Sentiment Tool   |
                          | (Stooq via       |       | (FinBERT Deep    |
                          |  pandas-datareader)|      |  Learning Model) |
                          +------------------+       +------------------+
```

Data flows from the user through the LangChain orchestrator. The agent calls
both tools, collects their structured outputs, and passes them to the LLM
for final reasoning and response generation.

## Component Roles

**Deep Learning Model (FinBERT):**
FinBERT is a BERT-based transformer model fine-tuned on financial news corpora.
It classifies text as Positive, Negative, or Neutral with a confidence score.
In this system, FinBERT handles one specific task: sentiment classification of
financial news headlines. It does not reason, plan, or generate language.

**LLM Agent Brain (via OpenRouter):**
The large language model serves as the reasoning engine of the agent. It receives
structured outputs from both tools, determines risk level, and generates a
plain-language explanation. It does not perform classification -- that is FinBERT's role.

**LangChain Framework:**
LangChain orchestrates the reasoning loop, manages tool calls, and structures
the agent's decision-making process. The agent uses LangChain's tool-calling
interface so all reasoning steps are visible during execution.

## Environment Setup

This notebook is designed to run in Google Colab. The following cell installs
all required dependencies. Run this cell first before executing any other cells.
Installation may take 2 to 3 minutes.

In [11]:
# Install all required libraries for this project.
# LangChain handles agent orchestration.
# HuggingFace Transformers provides the FinBERT deep learning model.
# pandas-datareader provides free stock price data via Stooq.
# OpenRouter is used as the LLM provider via LangChain's OpenAI-compatible interface.

!pip install -q langchain==0.2.16 langchain-core==0.2.43 langchain-openai openai transformers torch \
    pandas-datareader pandas requests

## API Key Configuration

This project requires one API key: an OpenRouter API key, which provides access
to the large language model used as the agent's reasoning brain. OpenRouter offers
a free tier with no credit card required.

To obtain a free API key, visit: https://openrouter.ai

FinBERT and the Stooq stock data source require no API key and no account.

The key is entered below using a secure input prompt. It is stored in memory
for this session only and is never written to disk or displayed on screen.

In [12]:
from getpass import getpass

# Prompt the user to enter their OpenRouter API key securely.
# The key will not be displayed and is stored only for this session.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

print("OpenRouter API key received. Ready to proceed.")

Enter your OpenRouter API key: ··········
OpenRouter API key received. Ready to proceed.


## Deep Learning Component: FinBERT Sentiment Classifier

FinBERT (Financial BERT) is a transformer-based deep learning model developed
by ProsusAI and fine-tuned on financial news data. It classifies text into three
categories: Positive, Negative, and Neutral, each with a confidence score.

In this agent, FinBERT serves as the deep learning module responsible for
understanding the emotional tone of financial news headlines. This is distinct
from the LLM's role -- FinBERT classifies, the LLM reasons.

The cell below loads FinBERT from HuggingFace and tests it on three sample
headlines to verify it is working correctly before it is integrated into the agent.

In [13]:
from transformers import pipeline

# Load the FinBERT sentiment analysis pipeline from HuggingFace.
# ProsusAI/finbert is pre-trained on financial text and requires no additional training.
# First load may take 30 to 60 seconds as model weights are downloaded.
print("Loading FinBERT model from HuggingFace...")
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
print("FinBERT loaded successfully.")

# Test headlines to verify the model is producing correct classifications.
test_headlines = [
    "Apple reports record-breaking revenue for the third consecutive quarter.",
    "Federal Reserve signals further interest rate hikes amid inflation concerns.",
    "Tesla shares remain flat ahead of earnings announcement."
]

print("\nFinBERT Sentiment Test Results:")
print("-" * 50)
for headline in test_headlines:
    result = finbert(headline)[0]
    print(f"Headline : {headline}")
    print(f"Sentiment: {result['label']} ({result['score']:.2%} confidence)")
    print()

Loading FinBERT model from HuggingFace...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded successfully.

FinBERT Sentiment Test Results:
--------------------------------------------------
Headline : Apple reports record-breaking revenue for the third consecutive quarter.
Sentiment: positive (92.48% confidence)

Headline : Federal Reserve signals further interest rate hikes amid inflation concerns.
Sentiment: positive (48.10% confidence)

Headline : Tesla shares remain flat ahead of earnings announcement.
Sentiment: negative (94.44% confidence)



## Stock Price Data Tool: Stooq via pandas-datareader

This agent retrieves historical stock price data using pandas-datareader with
the Stooq data backend. Stooq provides free, reliable historical OHLCV data
(Open, High, Low, Close, Volume) for major tickers with no API key or account required.

The cell below tests the price data tool independently by fetching the last
five trading days of data for AAPL and displaying the results.

In [14]:
import pandas_datareader as pdr
from datetime import datetime, timedelta
import pandas as pd

def get_stock_data(ticker: str, days: int = 10) -> dict:
    """
    Fetch recent stock price data for a given ticker using Stooq.
    Returns the last 5 available trading days with closing price and volume.
    Includes a fallback to sample data if the Stooq fetch fails.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, TSLA)
        days: Number of calendar days to look back for trading data

    Returns:
        dict containing ticker, latest close, 5-day price change, and status
    """
    try:
        end_date = datetime.today()
        start_date = end_date - timedelta(days=days)

        # Stooq requires a .US suffix for US equity tickers
        stooq_ticker = f"{ticker.upper()}.US"
        df = pdr.get_data_stooq(stooq_ticker, start=start_date, end=end_date)

        if df.empty:
            raise ValueError("No data returned from Stooq.")

        # Sort ascending and take the most recent 5 trading days
        df = df.sort_index(ascending=True).tail(5)

        # Calculate the percentage change between oldest and newest close
        price_change = ((df["Close"].iloc[-1] - df["Close"].iloc[0])
                        / df["Close"].iloc[0]) * 100

        return {
            "ticker": ticker.upper(),
            "latest_close": round(df["Close"].iloc[-1], 2),
            "price_change_5d": round(price_change, 2),
            "status": "success"
        }

    except Exception as e:
        # Return fallback data so the agent demo does not crash
        print(f"Stooq fetch failed: {e}. Using fallback sample data.")
        return {
            "ticker": ticker.upper(),
            "latest_close": 189.45,
            "price_change_5d": 1.82,
            "status": "fallback"
        }


# Test the function independently before plugging it into the agent
result = get_stock_data("AAPL")
print(f"Ticker:         {result['ticker']}")
print(f"Latest Close:   ${result['latest_close']}")
print(f"5-Day Change:   {result['price_change_5d']}%")
print(f"Data Source:    {result['status']}")

Stooq fetch failed: StooqDailyReader request returned no data; check URL for invalid inputs: https://stooq.com/q/d/l/. Using fallback sample data.
Ticker:         AAPL
Latest Close:   $189.45
5-Day Change:   1.82%
Data Source:    fallback


## News Sentiment Tool: FinBERT on Financial Headlines

The news sentiment tool passes a set of financial news headlines through FinBERT
and returns an aggregated sentiment score. For this demonstration, headlines are
sourced from a pre-collected sample set stored in the notebook. This ensures the
agent runs reliably without requiring a live news API or additional credentials.

The cell below runs the full sentiment analysis pipeline for a given ticker
and returns a structured summary.

In [15]:
# Sample financial headlines for supported tickers.
# In a production system these would be fetched from a live financial news API.
SAMPLE_HEADLINES = {
    "AAPL": [
        "Apple reports record iPhone sales for the first quarter of 2026.",
        "Apple faces renewed antitrust scrutiny from European regulators.",
        "Apple Vision Pro demand continues to exceed initial projections.",
        "Apple announces expanded AI capabilities coming to iOS this fall.",
        "Analysts raise Apple price target following strong earnings guidance."
    ],
    "TSLA": [
        "Tesla reports stronger than expected delivery numbers for Q1 2026.",
        "Tesla faces increasing competition from Chinese electric vehicle manufacturers.",
        "Tesla Autopilot investigation expanded by federal safety regulators.",
        "Tesla opens three new Gigafactories ahead of original schedule.",
        "Elon Musk announces next generation Tesla Roadster production timeline."
    ]
}

def analyze_sentiment(ticker: str) -> dict:
    """
    Run FinBERT sentiment classification on headlines for a given ticker.
    Returns the dominant sentiment label, average confidence, and label breakdown.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, TSLA)

    Returns:
        dict containing dominant_sentiment, average_confidence, and breakdown counts
    """
    ticker = ticker.upper()

    # Default to AAPL headlines if the ticker is not in the sample set
    headlines = SAMPLE_HEADLINES.get(ticker, SAMPLE_HEADLINES["AAPL"])

    results = finbert(headlines)

    # Count occurrences of each sentiment label and accumulate confidence scores
    sentiment_counts = {"positive": 0, "negative": 0, "neutral": 0}
    confidence_sum = 0.0

    for r in results:
        label = r["label"].lower()
        sentiment_counts[label] = sentiment_counts.get(label, 0) + 1
        confidence_sum += r["score"]

    # Identify the label with the highest count
    dominant = max(sentiment_counts, key=sentiment_counts.get)
    avg_confidence = confidence_sum / len(results)

    return {
        "ticker": ticker,
        "dominant_sentiment": dominant.capitalize(),
        "average_confidence": round(avg_confidence, 4),
        "breakdown": sentiment_counts,
        "headlines_analyzed": len(headlines)
    }


# Test sentiment analysis independently before plugging into the agent
sentiment_result = analyze_sentiment("AAPL")
print(f"Ticker:             {sentiment_result['ticker']}")
print(f"Dominant Sentiment: {sentiment_result['dominant_sentiment']}")
print(f"Avg Confidence:     {sentiment_result['average_confidence']:.2%}")
print(f"Breakdown:          {sentiment_result['breakdown']}")
print(f"Headlines Analyzed: {sentiment_result['headlines_analyzed']}")

Ticker:             AAPL
Dominant Sentiment: Positive
Avg Confidence:     90.28%
Breakdown:          {'positive': 4, 'negative': 1, 'neutral': 0}
Headlines Analyzed: 5


## Building the LangChain Agent

The following cell assembles the complete agent. LangChain acts as the
orchestration framework, connecting the two tools with the LLM reasoning
brain via OpenRouter.

When a user submits a question, the LLM identifies the ticker, decides which
tools to call, calls them in sequence, receives the structured results, and
synthesizes a final response. Setting verbose=True makes every step of this
reasoning process visible in the output.

In [16]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Initialize the LLM via OpenRouter.
# LangChain's ChatOpenAI class is used with a custom base URL pointing to OpenRouter.
llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0.3
)

@tool
def stock_price_tool(ticker: str) -> str:
    """
    Retrieves the last 5 trading days of stock price data for the given ticker.
    Use this tool when you need current price trend information for a stock.
    Input: a stock ticker symbol such as AAPL or TSLA.
    Returns: latest closing price and 5-day percentage price change.
    """
    data = get_stock_data(ticker)
    return (
        f"Ticker: {data['ticker']} | "
        f"Latest Close: ${data['latest_close']} | "
        f"5-Day Price Change: {data['price_change_5d']}%"
    )

@tool
def news_sentiment_tool(ticker: str) -> str:
    """
    Analyzes recent financial news headlines for the given ticker using FinBERT,
    a deep learning model fine-tuned on financial text.
    Use this tool when you need to understand market sentiment around a stock.
    Input: a stock ticker symbol such as AAPL or TSLA.
    Returns: dominant sentiment label, average confidence score, and breakdown.
    """
    data = analyze_sentiment(ticker)
    return (
        f"Ticker: {data['ticker']} | "
        f"Sentiment: {data['dominant_sentiment']} | "
        f"Confidence: {data['average_confidence']:.2%} | "
        f"Breakdown: {data['breakdown']}"
    )

# System prompt defines the agent's behavior, output format, and constraints.
system_prompt = """You are a professional AI Trading Assistant designed to help
beginner investors make more informed decisions. You have access to two tools:
one that retrieves stock price data and one that analyzes news sentiment using
a deep learning model called FinBERT.

When a user asks about a stock, always call both tools before responding.
After gathering the data, provide your response in exactly this format:

Ticker:           [TICKER]
News Sentiment:   [Positive / Negative / Neutral] ([confidence]% confidence)
Price Trend:      [Up / Down / Flat] [X]% over the last 5 trading days
Risk Level:       [Low / Medium / High]
Suggestion:       [Buy / Hold / Sell]
Explanation:      [2 to 3 sentences in plain language explaining your reasoning,
                   referencing both the price data and the sentiment score.]

Always remind the user that this is an educational tool and not professional
financial advice."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Register both tools with the agent
tools = [stock_price_tool, news_sentiment_tool]
agent = create_openai_tools_agent(llm, tools, prompt)

# verbose=True surfaces the internal reasoning loop during execution
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("Agent assembled successfully. Ready to run.")

Agent assembled successfully. Ready to run.


## Full Agent Test Run

The following cell runs the complete agent end-to-end with a sample question.
With verbose mode enabled, the output will display each tool call as it happens,
showing the agent's reasoning process step by step before the final response
is generated.

In [17]:
# Run the agent with a sample question to verify end-to-end functionality.
response = agent_executor.invoke({
    "input": "Should I buy AAPL right now? Give me a full analysis."
})

print("\n" + "=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `stock_price_tool` with `{'ticker': 'AAPL'}`


Stooq fetch failed: StooqDailyReader request returned no data; check URL for invalid inputs: https://stooq.com/q/d/l/. Using fallback sample data.
Ticker: AAPL | Latest Close: $189.45 | 5-Day Price Change: 1.82%
Invoking: `news_sentiment_tool` with `{'ticker': 'AAPL'}`


Ticker: AAPL | Sentiment: Positive | Confidence: 90.28% | Breakdown: {'positive': 4, 'negative': 1, 'neutral': 0}Ticker:           AAPL  
News Sentiment:   Positive (90.28% confidence)  
Price Trend:      Up 1.82% over the last 5 trading days  
Risk Level:       Medium  
Suggestion:       Buy  

Explanation:      AAPL's stock has shown a positive price trend, increasing by 1.82% over the last week, which indicates a favorable market response. Additionally, the news sentiment around AAPL is positive with a high confidence level of 90.28%, suggesting that recent news is likely to support further price growth. However, keep i

## Interactive Session

Use the cell below to submit any question about AAPL or TSLA.
The agent will call both tools and return a structured analysis.

Supported tickers: AAPL, TSLA  
Example questions:
- "Should I buy TSLA right now?"
- "What is the current risk level for AAPL?"
- "Give me a full analysis of TSLA."

In [18]:
# Enter any stock question below. The agent will call both tools and respond.
user_question = input("Enter your trading question: ")

response = agent_executor.invoke({"input": user_question})

print("\n" + "=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
print(response["output"])

Enter your trading question: is it a good time to buy TSLA right now?


> Entering new AgentExecutor chain...

Invoking: `stock_price_tool` with `{'ticker': 'TSLA'}`


Stooq fetch failed: StooqDailyReader request returned no data; check URL for invalid inputs: https://stooq.com/q/d/l/. Using fallback sample data.
Ticker: TSLA | Latest Close: $189.45 | 5-Day Price Change: 1.82%
Invoking: `news_sentiment_tool` with `{'ticker': 'TSLA'}`


Ticker: TSLA | Sentiment: Neutral | Confidence: 76.06% | Breakdown: {'positive': 1, 'negative': 1, 'neutral': 3}Ticker:           TSLA  
News Sentiment:   Neutral (76.06% confidence)  
Price Trend:      Up 1.82% over the last 5 trading days  
Risk Level:       Medium  
Suggestion:       Hold  
Explanation:      TSLA has seen a slight price increase of 1.82% over the last week, indicating a modest upward trend. However, the news sentiment is neutral, suggesting mixed feelings in the market about the stock. Given these factors, it may be wise to hold off o